# Collaborative Filtering Baselines

## Imports

In [1]:
import json
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import display

from helpers import create_model_name

# Teammate inputs: Task 2's collaborative filtering, refitted here on the holdout splits.
# Its canonical-ID refactor imports shared_data from the repository root, so the root goes on
# the path as well as task2 itself.
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath(os.path.join("..", "task2")))
from utils.biased_mf import (
    fit_biased_mf, load_biased_mf, predict_biased_mf, save_biased_mf)
from utils.drug_cf import fit_drug_cf, load_drug_cf, predict_drug_cf, save_drug_cf
from utils.evaluation import mean_squared_error
from utils.protein_cf import fit_protein_cf, load_protein_cf, predict_protein_cf, save_protein_cf

## Config

In [2]:
PROJECT_ROOT = os.path.abspath("..")

# Inputs, written by prepare_data.ipynb
DATA_DIR = os.path.join(PROJECT_ROOT, "open_task", "data")
SPLIT_PATTERN = os.path.join(DATA_DIR, "protein-holdout-{holdout}", "{dataset}_{split}.csv")
DRUG_MAPPING_FILE = os.path.join(DATA_DIR, "drug_mapping.csv")
PROTEIN_MAPPING_FILE = os.path.join(DATA_DIR, "protein_mapping.csv")
DRUG_ID_COLUMN = "drug_id"
PROTEIN_ID_COLUMN = "protein_id"
SMILES_COLUMN = "compound_iso_smiles"
SEQUENCE_COLUMN = "target_sequence"
AFFINITY_COLUMN = "affinity"
PREDICTION_COLUMN = "prediction"
FALLBACK_COLUMN = "fallback"

DATASETS = ["kiba", "davis"]
HOLDOUTS = [0.0, 0.1, 0.2, 0.3, 0.5, 0.75]
SPLITS = ("val", "test")
TRAIN_SPLIT = "train"
SELECTION_SPLIT = "val"

# Stage flag
FIT = True

# Task 2's grid, re-run per holdout so each ratio gets its own best neighbourhood
SHRINKAGE = 10.0
NEIGHBOUR_GRID = [5, 10, 20, 30]
MIN_COMMON_GRID = [2, 5, 10]
SCORE_SCALES = {
    "davis": "pKd; supplied values, no additional log transformation",
    "kiba": "supplied transformed KIBA scores",
}

# Task 2's selected biased MF configuration, refitted here per holdout
BIASED_MF = {
    "n_factors": 128,
    "learning_rate": 0.001,
    "regularization": 1.0,
    # Patience ends a converged run; DAVIS is still improving at 100, so the cap is only
    # a safety limit, raised to the ceiling Task 2 used for its own extended runs
    "max_epochs": 300,
    "patience": 10,
    "seed": 42,
}

# Outputs
MODEL_DIR = os.path.join(PROJECT_ROOT, "open_task", "models", "saved_models")
MODEL_PATTERN = os.path.join(MODEL_DIR, "{model}_{name}.npz")
GLOBAL_MEAN_PATTERN = os.path.join(MODEL_DIR, "global_mean_{name}.json")
PREDICTION_DIR = os.path.join(PROJECT_ROOT, "open_task", "predictions")
PREDICTION_PATTERN = os.path.join(PREDICTION_DIR, "{model}_{name}_{split}.csv")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PREDICTION_DIR, exist_ok=True)

pd.set_option("display.float_format", "{:.4f}".format)

## Splits

In [3]:
# keep_default_na=False so a SMILES like NA stays a string
drug_mapping = pd.read_csv(DRUG_MAPPING_FILE, keep_default_na=False)
protein_mapping = pd.read_csv(PROTEIN_MAPPING_FILE, keep_default_na=False)
smiles = dict(zip(drug_mapping[DRUG_ID_COLUMN], drug_mapping[SMILES_COLUMN]))
sequences = dict(zip(protein_mapping[PROTEIN_ID_COLUMN], protein_mapping[SEQUENCE_COLUMN]))


def read_holdout(dataset, holdout, split):
    """A holdout split with the text columns the collaborative filtering code keys on."""
    frame = pd.read_csv(SPLIT_PATTERN.format(dataset=dataset, holdout=holdout, split=split))
    frame[SMILES_COLUMN] = frame[DRUG_ID_COLUMN].map(smiles)
    frame[SEQUENCE_COLUMN] = frame[PROTEIN_ID_COLUMN].map(sequences)
    return frame


splits = {
    (dataset, holdout, split): read_holdout(dataset, holdout, split)
    for dataset in DATASETS
    for holdout in HOLDOUTS
    for split in (TRAIN_SPLIT, *SPLITS)
}

display(pd.DataFrame(
    [
        {"dataset": dataset, "holdout": holdout, "split": split, "pairs": len(frame)}
        for (dataset, holdout, split), frame in splits.items()
    ]
).set_index(["dataset", "holdout", "split"]))

def training_axis_ids(dataset, holdout):
    """Renumber both axes so the training split covers 0..n-1, which the CF code requires.

    Task 2's canonical-ID CF and MF reject a training frame whose IDs are not contiguous from
    zero, and holding proteins out of training is exactly that. Renumbering per holdout
    restores it: training entities keep their relative order, and anything absent from
    training is numbered above its axis, so positions() reads it as unknown and the model
    falls back on a mean as it did before.
    """
    train = splits[dataset, holdout, TRAIN_SPLIT]
    renumbered = {}

    for column in (DRUG_ID_COLUMN, PROTEIN_ID_COLUMN):
        seen = np.sort(train[column].unique())
        everywhere = np.unique(np.concatenate(
            [splits[dataset, holdout, split][column].to_numpy() for split in SPLITS]))
        order = np.concatenate([seen, everywhere[~np.isin(everywhere, seen)]])
        renumbered[column] = pd.Series(np.arange(len(order)), index=order)

    return renumbered


def on_training_axis(frame, axis_ids):
    """The same frame with both ID columns renumbered onto those training axes."""
    return frame.assign(**{
        column: mapping.reindex(frame[column]).to_numpy(dtype=np.int64)
        for column, mapping in axis_ids.items()
    })


pairs
dataset holdout split       
kiba    0.0000  train  82359
                val    17648
                test   17650
        0.1000  train  74013
                val    17648
                test   17650
        0.2000  train  64867
                val    17648
                test   17650
        0.3000  train  57698
                val    17648
                test   17650
        0.5000  train  43502
                val    17648
                test   17650
        0.7500  train  22311
                val    17648
                test   17650
davis   0.0000  train  18040
                val     3865
                test    3867
        0.1000  train  16225
                val     3865
                test    3867
        0.2000  train  14434
                val     3865
                test    3867
        0.3000  train  12640
                val     3865
                test    3867
        0.5000  train   9021
                val     3865
                test    3867
        0.7500  train   4511
                val     3865
                test    3867

## Fitting

In [4]:
def select_neighbourhood(model, predict, validation):
    """Task 2's grid: lowest validation MSE, ties broken by ascending k then min_common."""
    pairs = validation[[DRUG_ID_COLUMN, PROTEIN_ID_COLUMN]]
    trials = [
        {
            "k": k,
            "min_common": minimum,
            "val_mse": mean_squared_error(
                validation[AFFINITY_COLUMN],
                predict(model, pairs, k=k, min_common=minimum)[0]),
        }
        for k in NEIGHBOUR_GRID
        for minimum in MIN_COMMON_GRID
    ]
    best = sorted(trials, key=lambda trial: (trial["val_mse"], trial["k"], trial["min_common"]))[0]
    return best["k"], best["min_common"]


def fitted(dataset, holdout):
    """The three baselines for one holdout, refitted on its training split or loaded back."""
    name = create_model_name(dataset, holdout)
    paths = {
        "drug_cf": MODEL_PATTERN.format(model="drug_cf", name=name),
        "protein_cf": MODEL_PATTERN.format(model="protein_cf", name=name),
        "biased_mf": MODEL_PATTERN.format(model="biased_mf", name=name),
        "global_mean": GLOBAL_MEAN_PATTERN.format(name=name),
    }

    if not FIT:
        with open(paths["global_mean"]) as stream:
            return {
                "drug_cf": load_drug_cf(paths["drug_cf"]),
                "protein_cf": load_protein_cf(paths["protein_cf"]),
                "biased_mf": load_biased_mf(paths["biased_mf"]),
                "global_mean": json.load(stream),
            }

    axis_ids = training_axis_ids(dataset, holdout)
    train = on_training_axis(splits[dataset, holdout, TRAIN_SPLIT], axis_ids)
    validation = on_training_axis(splits[dataset, holdout, SELECTION_SPLIT], axis_ids)
    n_drugs = int(train[DRUG_ID_COLUMN].max()) + 1
    n_proteins = int(train[PROTEIN_ID_COLUMN].max()) + 1
    shared = {"dataset": dataset, "score_scale": SCORE_SCALES[dataset]}

    built = {
        "drug_cf": (fit_drug_cf(train, SHRINKAGE), predict_drug_cf, save_drug_cf),
        "protein_cf": (fit_protein_cf(train, SHRINKAGE), predict_protein_cf, save_protein_cf),
    }
    models = {}

    for model_name, (model, predict, save) in built.items():
        # fit_* records neither k nor min_common, and predict_* reads both off the model
        k, minimum = select_neighbourhood(model, predict, validation)
        models[model_name] = {**model, **shared, "k": k, "min_common": minimum}
        save(models[model_name], paths[model_name])

    # fit_biased_mf scores every validation row to choose an epoch and raises on an ID it
    # never trained on, so epoch selection is given the rows it can score
    known = (
        (validation[DRUG_ID_COLUMN] < n_drugs)
        & (validation[PROTEIN_ID_COLUMN] < n_proteins)
    )
    models["biased_mf"] = fit_biased_mf(train, validation[known], **BIASED_MF, **shared)
    save_biased_mf(models["biased_mf"], paths["biased_mf"])

    models["global_mean"] = {
        "model": "Global Mean",
        "holdout": holdout,
        "training_mean": float(train[AFFINITY_COLUMN].mean()),
        "training_observations": len(train),
        **shared,
    }
    with open(paths["global_mean"], "w") as stream:
        json.dump(models["global_mean"], stream, indent=1)

    return models


baselines = {}

for dataset in DATASETS:
    for holdout in HOLDOUTS:
        baselines[dataset, holdout] = fitted(dataset, holdout)
        print(f"{create_model_name(dataset, holdout)} fitted")

display(pd.DataFrame(
    [
        {
            "dataset": dataset,
            "holdout": holdout,
            "model": model_name,
            "k": int(model["k"]),
            "min_common": int(model["min_common"]),
        }
        for (dataset, holdout), models in baselines.items()
        for model_name, model in models.items()
        if model_name in ("drug_cf", "protein_cf")
    ]
).set_index(["dataset", "holdout", "model"]))

kiba_holdout_0.0 fitted
kiba_holdout_0.1 fitted
kiba_holdout_0.2 fitted
kiba_holdout_0.3 fitted
kiba_holdout_0.5 fitted
kiba_holdout_0.75 fitted
davis_holdout_0.0 fitted
davis_holdout_0.1 fitted
davis_holdout_0.2 fitted
davis_holdout_0.3 fitted
davis_holdout_0.5 fitted
davis_holdout_0.75 fitted


k  min_common
dataset holdout model                     
kiba    0.0000  drug_cf     10           2
                protein_cf   5           2
        0.1000  drug_cf     10           2
                protein_cf   5           2
        0.2000  drug_cf     10           2
                protein_cf   5           2
        0.3000  drug_cf     10           2
                protein_cf  10           2
        0.5000  drug_cf     10           2
                protein_cf   5          10
        0.7500  drug_cf     10           2
                protein_cf   5           2
davis   0.0000  drug_cf      5           2
                protein_cf  10           2
        0.1000  drug_cf      5           2
                protein_cf  10           2
        0.2000  drug_cf     10           2
                protein_cf  10           2
        0.3000  drug_cf     10           2
                protein_cf  10           2
        0.5000  drug_cf     10           2
                protein_cf   5           2
        0.7500  drug_cf     10           2
                protein_cf   5           2

## Predictions

In [5]:
summary = []

for (dataset, holdout), models in baselines.items():
    name = create_model_name(dataset, holdout)
    axis_ids = training_axis_ids(dataset, holdout)
    trained_proteins = set(splits[dataset, holdout, TRAIN_SPLIT][PROTEIN_ID_COLUMN])
    drug_means = splits[dataset, holdout, TRAIN_SPLIT].groupby(
        DRUG_ID_COLUMN)[AFFINITY_COLUMN].mean()
    n_drugs = len(models["biased_mf"]["drugs"])
    n_proteins = len(models["biased_mf"]["targets"])

    for split in SPLITS:
        frame = splits[dataset, holdout, split]
        renumbered = on_training_axis(frame, axis_ids)
        pairs = renumbered[[DRUG_ID_COLUMN, PROTEIN_ID_COLUMN]]
        cold = ~frame[PROTEIN_ID_COLUMN].isin(trained_proteins)

        # Biased MF raises on an unseen drug or protein instead of falling back. Only the
        # protein is held out, so those rows keep the drug's own training mean, the tier
        # drug CF already uses here; the global mean covers a drug absent from training
        known = (
            (renumbered[DRUG_ID_COLUMN] < n_drugs)
            & (renumbered[PROTEIN_ID_COLUMN] < n_proteins)
        ).to_numpy()
        biased = drug_means.reindex(frame[DRUG_ID_COLUMN]).fillna(
            models["global_mean"]["training_mean"]).to_numpy(copy=True)
        biased[known] = predict_biased_mf(models["biased_mf"], pairs[known])

        predicted = {
            "drug_cf": predict_drug_cf(models["drug_cf"], pairs),
            "protein_cf": predict_protein_cf(models["protein_cf"], pairs),
            "biased_mf": (biased, ~known),
            "global_mean": (
                np.full(len(frame), models["global_mean"]["training_mean"]),
                np.ones(len(frame), dtype=bool),
            ),
        }

        for model_name, (predictions, fallback) in predicted.items():
            frame.assign(**{PREDICTION_COLUMN: predictions, FALLBACK_COLUMN: fallback})[
                [DRUG_ID_COLUMN, PROTEIN_ID_COLUMN, PREDICTION_COLUMN, FALLBACK_COLUMN]
            ].to_csv(
                PREDICTION_PATTERN.format(model=model_name, name=name, split=split), index=False)

            summary.append({
                "dataset": dataset,
                "holdout": holdout,
                "split": split,
                "model": model_name,
                "pairs": len(frame),
                "cold rows": int(cold.sum()),
                "fallback rows": int(fallback.sum()),
                "MSE": mean_squared_error(frame[AFFINITY_COLUMN], predictions),
            })

display(pd.DataFrame(summary).set_index(["dataset", "holdout", "split", "model"]))

pairs  cold rows  fallback rows    MSE
dataset holdout split model                                              
kiba    0.0000  val   drug_cf      17648          0            449 0.2090
                      protein_cf   17648          0              1 0.2398
                      biased_mf    17648          0              0 0.1888
                      global_mean  17648          0          17648 0.6878
                test  drug_cf      17650          0            396 0.2187
...                                  ...        ...            ...    ...
davis   0.7500  val   global_mean   3865       2870           3865 0.6754
                test  drug_cf       3867       2913           2941 0.4685
                      protein_cf    3867       2913           2913 0.5874
                      biased_mf     3867       2913           2913 0.4607
                      global_mean   3867       2913           3867 0.6793

[96 rows x 4 columns]